# Compare clustering

We compare three clustering methods on two feature sets, because both choices matter:

* **vit layer 12** 
* **vit layer 3**

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster

from pam import pam, labels as pam_labels

DATA = "../week_2/data"

d = np.load(f"{DATA}/embeddings_stl10_1000.npz")

y = d["y"]

FEATURES = {
    "vit layer 12": d["E_l12"],
    "vit layer 3": d["E_l3"],
}

K = 10
RANDOM_STATE = 0

print({k: v.shape for k, v in FEATURES.items()})

In [ ]:
rows = []

for fname, X in FEATURES.items():

    # --- k-means -------------------------------------------------
    t0 = time.perf_counter()
    km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)
    t_km = time.perf_counter() - t0

    rows.append({
        "features": fname, "method": "k-means",
        "ARI vs truth": adjusted_rand_score(y, km.labels_),
        "NMI": normalized_mutual_info_score(y, km.labels_),
        "silhouette": silhouette_score(X, km.labels_),
        "seconds": t_km,
    })

    # --- k-medoids -----------------------------------------------
    t0 = time.perf_counter()
    medoids, _ = pam(X, K, random_state=RANDOM_STATE)
    t_pam = time.perf_counter() - t0

    lb = pam_labels(X, medoids)

    rows.append({
        "features": fname, "method": "k-medoids",
        "ARI vs truth": adjusted_rand_score(y, lb),
        "NMI": normalized_mutual_info_score(y, lb),
        "silhouette": silhouette_score(X, lb),
        "seconds": t_pam,
    })

    # --- hierarchical --------------------------------------------
    t0 = time.perf_counter()
    Z = linkage(X, method="ward")
    lb = fcluster(Z, t=K, criterion="maxclust")
    t_h = time.perf_counter() - t0

    rows.append({
        "features": fname, "method": "hierarchical",
        "ARI vs truth": adjusted_rand_score(y, lb),
        "NMI": normalized_mutual_info_score(y, lb),
        "silhouette": silhouette_score(X, lb),
        "seconds": t_h,
    })

results = (pd.DataFrame(rows)
           .sort_values(["features", "method"])
           .reset_index(drop=True))

In [ ]:
results.style.format({
    "ARI vs truth": "{:.3f}",
    "NMI": "{:.3f}",
    "silhouette": "{:.3f}",
    "seconds": "{:.2f}",
}).background_gradient(subset=["ARI vs truth"], cmap="Blues")